In [1]:
!apt-get install -y -q ffmpeg
!pip install -q \
    chromadb \
    sentence-transformers \
    faster-whisper \
    transformers \
    Pillow \
    moviepy \
    fastapi \
    uvicorn \
    nest-asyncio \
    pyngrok \
    gradio \
    torch torchvision \
    accelerate

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [2]:
# Run this in a Colab cell to create the folder and upload index.html
import os
os.makedirs("/content/ui", exist_ok=True)

# Then manually upload index.html:
# Left panel → Files icon → upload → drag index.html into /content/ui/

In [3]:
import os, io, base64, shutil, time, threading
import numpy as np
from pathlib import Path
from PIL import Image

In [4]:
# ── Embedding model ──────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

print("Loading embedding model …")
EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("  ✓ all-MiniLM-L6-v2 ready")

# ── Whisper STT ───────────────────────────────────────────────────────────────
from faster_whisper import WhisperModel

print("Loading Whisper …")
# use 'base' on CPU, 'small' on GPU for better accuracy
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
COMPUTE = "float16" if DEVICE == "cuda" else "int8"
WHISPER  = WhisperModel("base", device=DEVICE, compute_type=COMPUTE)
print(f"  ✓ Whisper ready  (device={DEVICE})")

# ── BLIP image captioning ─────────────────────────────────────────────────────
from transformers import BlipProcessor, BlipForConditionalGeneration
import torch

print("Loading BLIP captioning model …")
BLIP_PROCESSOR = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
BLIP_MODEL = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(DEVICE)
print("  ✓ BLIP ready")

# ── ChromaDB ──────────────────────────────────────────────────────────────────
import chromadb

CHROMA_DIR = "/content/chroma_store"
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
COLLECTION   = chroma_client.get_or_create_collection(
    name="multimodal_store",
    metadata={"hnsw:space": "cosine"}   # cosine similarity
)
print(f"  ✓ ChromaDB ready  ({COLLECTION.count()} docs already stored)")
print("\nAll models loaded. Ready to ingest!")



Loading embedding model …


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  ✓ all-MiniLM-L6-v2 ready
Loading Whisper …
  ✓ Whisper ready  (device=cpu)
Loading BLIP captioning model …


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

  ✓ BLIP ready
  ✓ ChromaDB ready  (0 docs already stored)

All models loaded. Ready to ingest!


In [5]:
# =============================================================================
# CELL 3 — Ingestion pipelines  (text · audio · video)
# =============================================================================

# ── Shared helpers ─────────────────────────────────────────────────────────────

def embed(text: str) -> list:
    """Embed a string with MiniLM and return a plain Python list."""
    return EMBED_MODEL.encode(text, normalize_embeddings=True).tolist()


def safe_add(doc_id: str, embedding: list, document: str, metadata: dict):
    """Add to ChromaDB; skip silently if the id already exists."""
    try:
        COLLECTION.add(
            ids=[doc_id],
            embeddings=[embedding],
            documents=[document],
            metadatas=[metadata]
        )
    except Exception:
        # duplicate id → upsert instead
        COLLECTION.upsert(
            ids=[doc_id],
            embeddings=[embedding],
            documents=[document],
            metadatas=[metadata]
        )


# ── TEXT pipeline ──────────────────────────────────────────────────────────────

def chunk_text(text: str, chunk_size: int = 400, overlap: int = 50):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks


def ingest_text(filepath: str):
    """Ingest a .txt or .md file into ChromaDB."""
    filepath = str(filepath)
    print(f"[TEXT] ingesting  {filepath}")
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()

    chunks = chunk_text(raw)
    fname  = Path(filepath).name

    for i, chunk in enumerate(chunks):
        chunk = chunk.strip()
        if not chunk:
            continue
        emb    = embed(chunk)
        doc_id = f"text__{fname}__chunk_{i}"
        safe_add(doc_id, emb, chunk, {
            "modality":    "text",
            "source":      filepath,
            "chunk_index": i,
            "filename":    fname,
        })

    print(f"  ✓ stored {len(chunks)} chunks")
    return len(chunks)


# ── AUDIO pipeline ─────────────────────────────────────────────────────────────

def transcribe(audio_path: str):
    """Return (full_text, segments) using faster-whisper."""
    segments_iter, info = WHISPER.transcribe(audio_path, beam_size=5)
    segments = list(segments_iter)          # consume the generator
    full_text = " ".join(s.text for s in segments)
    return full_text, segments


def ingest_audio(filepath: str):
    """Ingest a .mp3/.wav/.m4a file into ChromaDB."""
    filepath = str(filepath)
    print(f"[AUDIO] ingesting  {filepath}")
    fname = Path(filepath).name

    _, segments = transcribe(filepath)
    count = 0

    for i, seg in enumerate(segments):
        text = seg.text.strip()
        if not text:
            continue
        emb    = embed(text)
        doc_id = f"audio__{fname}__seg_{i}"
        safe_add(doc_id, emb, text, {
            "modality":   "audio",
            "source":     filepath,
            "filename":   fname,
            "start_time": round(seg.start, 2),
            "end_time":   round(seg.end,   2),
        })
        count += 1

    print(f"  ✓ stored {count} audio segments")
    return count


# ── VIDEO pipeline ─────────────────────────────────────────────────────────────

def caption_frame(pil_image: Image.Image) -> str:
    """Generate a caption for a PIL image using BLIP."""
    inputs = BLIP_PROCESSOR(images=pil_image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = BLIP_MODEL.generate(**inputs, max_new_tokens=60)
    return BLIP_PROCESSOR.decode(out[0], skip_special_tokens=True)


def extract_frames(video_path: str, every_n_sec: int = 3):
    """
    Use FFmpeg to extract one frame every `every_n_sec` seconds.
    Returns list of (timestamp_seconds, PIL.Image).
    """
    out_dir = f"/tmp/frames_{int(time.time())}"
    os.makedirs(out_dir, exist_ok=True)

    # FFmpeg: extract 1 frame every N seconds, output as JPEG
    cmd = (
        f"ffmpeg -i \"{video_path}\" "
        f"-vf fps=1/{every_n_sec} "
        f"{out_dir}/frame_%05d.jpg -y -loglevel quiet"
    )
    os.system(cmd)

    frames = []
    for idx, img_file in enumerate(sorted(Path(out_dir).glob("*.jpg"))):
        timestamp = idx * every_n_sec
        try:
            img = Image.open(img_file).convert("RGB")
            frames.append((timestamp, img))
        except Exception:
            pass

    shutil.rmtree(out_dir, ignore_errors=True)
    return frames


def extract_audio_from_video(video_path: str) -> str:
    """Extract audio track from video as a WAV file using FFmpeg."""
    out_path = f"/tmp/video_audio_{int(time.time())}.wav"
    cmd = (
        f"ffmpeg -i \"{video_path}\" "
        f"-ac 1 -ar 16000 -vn \"{out_path}\" "
        f"-y -loglevel quiet"
    )
    os.system(cmd)
    return out_path


def ingest_video(filepath: str, frame_every_sec: int = 3):
    """Ingest a .mp4/.mov/.avi file — both visual frames and audio track."""
    filepath = str(filepath)
    print(f"[VIDEO] ingesting  {filepath}")
    fname = Path(filepath).name
    total = 0

    # ── Visual stream: frame captions ──
    print("  Extracting frames …")
    frames = extract_frames(filepath, every_n_sec=frame_every_sec)
    for timestamp, pil_img in frames:
        caption = caption_frame(pil_img)
        emb     = embed(caption)
        doc_id  = f"video__{fname}__frame_{timestamp}s"
        safe_add(doc_id, emb, caption, {
            "modality":     "video",
            "source":       filepath,
            "filename":     fname,
            "content_type": "visual",
            "timestamp":    timestamp,
        })
        total += 1
    print(f"  ✓ stored {len(frames)} frame captions")

    # ── Audio stream: Whisper transcript ──
    print("  Transcribing audio track …")
    audio_path = extract_audio_from_video(filepath)
    if os.path.exists(audio_path):
        _, segments = transcribe(audio_path)
        for i, seg in enumerate(segments):
            text = seg.text.strip()
            if not text:
                continue
            emb    = embed(text)
            doc_id = f"video__{fname}__audio_seg_{i}"
            safe_add(doc_id, emb, text, {
                "modality":     "video",
                "source":       filepath,
                "filename":     fname,
                "content_type": "audio",
                "start_time":   round(seg.start, 2),
                "end_time":     round(seg.end,   2),
            })
            total += 1
        os.remove(audio_path)
        print(f"  ✓ stored {len(segments)} audio segments")

    print(f"  Total stored for this video: {total}")
    return total


# ── Router ─────────────────────────────────────────────────────────────────────

def ingest_file(filepath: str):
    """Detect modality from extension and call the right pipeline."""
    ext = Path(filepath).suffix.lower()
    if ext in {".txt", ".md"}:
        return ingest_text(filepath), "text"
    elif ext in {".mp3", ".wav", ".m4a", ".flac", ".ogg"}:
        return ingest_audio(filepath), "audio"
    elif ext in {".mp4", ".mov", ".avi", ".mkv", ".webm"}:
        return ingest_video(filepath), "video"
    else:
        raise ValueError(f"Unsupported file type: {ext}")

print("Ingestion pipelines ready.")


Ingestion pipelines ready.


In [6]:
# =============================================================================
# CELL 4 — Search engine
# =============================================================================

def search(query: str, n_results: int = 5, modality_filter: str = None):
    """
    Embed the query and return the top-N closest documents from ChromaDB.
    Optionally filter by modality: 'text' | 'audio' | 'video' | None (all).
    """
    query_emb = embed(query)

    where = {"modality": modality_filter} if modality_filter else None

    results = COLLECTION.query(
        query_embeddings=[query_emb],
        n_results=n_results,
        where=where,
        include=["documents", "metadatas", "distances"]
    )

    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        # cosine distance → similarity score 0-1
        score = round(1 - dist, 4)
        hits.append({
            "score":        score,
            "content":      doc,
            "modality":     meta.get("modality", "?"),
            "source":       meta.get("filename", meta.get("source", "")),
            "content_type": meta.get("content_type", ""),
            "timestamp":    meta.get("timestamp") or meta.get("start_time"),
        })

    return hits


def format_results(hits: list) -> str:
    """Return a human-readable string of search results."""
    if not hits:
        return "No results found."
    lines = []
    icons = {"text": "📄", "audio": "🎵", "video": "🎬"}
    for i, h in enumerate(hits, 1):
        icon  = icons.get(h["modality"], "📁")
        label = h["modality"].upper()
        if h["content_type"] == "visual":
            label += " (frame caption)"
        elif h["content_type"] == "audio":
            label += " (spoken)"
        ts = f"  ⏱ {h['timestamp']}s" if h["timestamp"] is not None else ""
        lines.append(
            f"{i}. {icon} [{label}]  score={h['score']}\n"
            f"   📂 {h['source']}{ts}\n"
            f"   💬 {h['content'][:200]}"
        )
    return "\n\n".join(lines)

print("Search engine ready.")


Search engine ready.


In [7]:
# Install PortAudio system library for sounddevice
!apt-get install -y -qq portaudio19-dev

In [8]:
# =============================================================================
# CELL 4.5 — Spoken query support (Audio → Text search)
# User speaks → Whisper transcribes → searches as text
# =============================================================================

# Install sounddevice if not already
!pip install -q sounddevice scipy

import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav
import tempfile, os

def record_and_search(duration_seconds=5, n_results=5):
    """
    Record microphone for N seconds → transcribe with Whisper → search.
    Returns (transcribed_text, search_hits)
    """
    SAMPLE_RATE = 16000
    print(f"🎙️  Recording for {duration_seconds} seconds... SPEAK NOW")

    audio_data = sd.rec(
        int(duration_seconds * SAMPLE_RATE),
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype='int16'
    )
    sd.wait()
    print("✓ Recording complete. Transcribing...")

    # Save to temp WAV
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    wav.write(tmp.name, SAMPLE_RATE, audio_data)

    # Transcribe with Whisper
    segments_iter, _ = WHISPER.transcribe(tmp.name, beam_size=5)
    segments = list(segments_iter)
    spoken_text = " ".join(s.text for s in segments).strip()
    os.unlink(tmp.name)

    if not spoken_text:
        print("✗ Could not detect speech. Try again.")
        return None, []

    print(f"✓ You said: \"{spoken_text}\"")
    print(f"Searching for: \"{spoken_text}\"..." )

    hits = search(spoken_text, n_results=n_results)
    return spoken_text, hits

print("Spoken query function ready.")

Spoken query function ready.


In [9]:
# In Colab — download a free sample video
!wget -q -O /content/sample_cooking.mp4 \
  "https://www.w3schools.com/html/mov_bbb.mp4"
print("✓ Video downloaded")

✓ Video downloaded


In [10]:
# Generate a video with spoken milk content
os.system("pip install -q gtts")
from gtts import gTTS

script = "A person is sitting at the kitchen table drinking a glass of cold milk. She picks up the milk carton and pours it into a white cup. The human drinks milk every morning for breakfast."

tts = gTTS(text=script, lang='en')
tts.save("/tmp/milk_audio.mp3")
os.system("ffmpeg -i /tmp/milk_audio.mp3 /tmp/milk_audio.wav -y -loglevel quiet")

os.system(
    "ffmpeg -y "
    "-f lavfi -i 'color=c=white:size=640x360:rate=24' "
    "-i /tmp/milk_audio.wav "
    "-shortest -c:v libx264 -c:a aac "
    "/content/human_drinking_milk.mp4 -loglevel quiet"
)
print("✓ Video ready: /content/human_drinking_milk.mp4")

✓ Video ready: /content/human_drinking_milk.mp4


In [11]:
# ── STEP 1: Check what's actually in the database ────────────────────────────
print(f"Docs in COLLECTION: {COLLECTION.count()}")

# Peek at what's stored
if COLLECTION.count() > 0:
    sample = COLLECTION.peek(limit=3)
    print("Sample docs stored:")
    for doc, meta in zip(sample['documents'], sample['metadatas']):
        print(f"  [{meta.get('modality')}] {meta.get('filename')} → {doc[:80]}")
else:
    print("COLLECTION is empty — this is the bug")

Docs in COLLECTION: 0
COLLECTION is empty — this is the bug


In [12]:
# ── STEP 2: Re-ingest directly into the correct COLLECTION ───────────────────

# First verify the file exists
import os
if os.path.exists("/content/human_drinking_milk.mp4"):
    size = os.path.getsize("/content/human_drinking_milk.mp4") / 1024
    print(f"✓ File exists: {size:.1f} KB")
else:
    print("✗ File missing — recreating it...")
    from gtts import gTTS
    tts = gTTS(text="A person is drinking a glass of cold milk at the kitchen table. The human pours milk every morning for breakfast.", lang='en')
    tts.save("/tmp/milk_audio.mp3")
    os.system("ffmpeg -i /tmp/milk_audio.mp3 /tmp/milk_audio.wav -y -loglevel quiet")
    os.system(
        "ffmpeg -y "
        "-f lavfi -i 'color=c=white:size=640x360:rate=24' "
        "-i /tmp/milk_audio.wav "
        "-shortest -c:v libx264 -c:a aac "
        "/content/human_drinking_milk.mp4 -loglevel quiet"
    )
    print("✓ Video recreated")

# ── STEP 3: Ingest and immediately verify ────────────────────────────────────
print(f"\nBefore ingest: {COLLECTION.count()} docs")

count, mod = ingest_file("/content/human_drinking_milk.mp4")
print(f"Ingested: {count} chunks as {mod}")
print(f"After ingest: {COLLECTION.count()} docs")  # must be > 0 now

# ── STEP 4: Test search directly in Colab (bypass UI) ────────────────────────
hits = search("human drinking milk", n_results=3)
print(f"\nSearch test — {len(hits)} hits:")
for h in hits:
    print(f"  [{h['modality']}] score={h['score']}  {h['content'][:80]}")

✓ File exists: 124.8 KB

Before ingest: 0 docs
[VIDEO] ingesting  /content/human_drinking_milk.mp4
  Extracting frames …
  ✓ stored 5 frame captions
  Transcribing audio track …
  ✓ stored 3 audio segments
  Total stored for this video: 8
Ingested: 8 chunks as video
After ingest: 8 docs

Search test — 3 hits:
  [video] score=0.7309  The human drinks milk every morning for breakfast.
  [video] score=0.6902  A person is sitting at the kitchen table drinking a glass of cold milk.
  [video] score=0.5384  She picks up the milk carton and pours it into a white cup.


In [13]:
# Creates a spoken audio file about a song by "XYZ band"
from gtts import gTTS
import os

script = """
XYZ is a famous music band from the 1990s.
Their most popular song is called Midnight Dreams.
The lead singer of XYZ performed beautifully at the concert.
XYZ released their debut album in 1995 and became worldwide famous.
The song Midnight Dreams by XYZ reached number one on the charts.
"""

tts = gTTS(text=script, lang='en')
tts.save("/content/xyz_band_audio.mp3")
print(f"✓ Audio file created")

# Ingest it
count, mod = ingest_file("/content/xyz_band_audio.mp3")
print(f"✓ Ingested as {mod} — {count} segments")
print(f"✓ Total docs in DB: {COLLECTION.count()}")

✓ Audio file created
[AUDIO] ingesting  /content/xyz_band_audio.mp3
  ✓ stored 3 audio segments
✓ Ingested as audio — 3 segments
✓ Total docs in DB: 11


In [14]:
# Creates a text document with searchable content
text_content = """
Artificial intelligence is transforming modern technology.
Machine learning models learn patterns from large datasets.
Deep learning is a subset of machine learning using neural networks.
Python is the most popular programming language for AI development.
The recipe requires two eggs, one cup of flour and a pinch of salt.
The cat sat quietly on the warm windowsill watching birds outside.
Natural language processing allows computers to understand human speech.
Data science combines statistics, programming and domain knowledge.
"""

with open("/content/knowledge_base.txt", "w") as f:
    f.write(text_content)
print("✓ Text file created")

# Ingest it
count, mod = ingest_file("/content/knowledge_base.txt")
print(f"✓ Ingested as {mod} — {count} chunks")
print(f"✓ Total docs in DB: {COLLECTION.count()}")

✓ Text file created
[TEXT] ingesting  /content/knowledge_base.txt
  ✓ stored 2 chunks
✓ Ingested as text — 2 chunks
✓ Total docs in DB: 13


In [15]:
# Creates a text document with searchable content
text_content = """
One, two, three
Why are you hanging on so tight
To the rope that I'm hanging from?
Off this island, this was an escape plan (this was an escape plan)
Carefully timed it, so let me go
And dive into the waves below
Who tends the orchards? Who fixes up the gables?
Emotional torture from the head of your high table
Who fetches the water from the rocky mountain spring?
And walk back down again to feel your words
And their sharp sting
And I'm getting fucking tired
The capillaries in my eyes are bursting
If our love died, would that be the worst thing?
For somebody I thought was my saviour
You sure make me do a whole lot of labour
The calloused skin on my hands is cracking
If our love ends, would that be a bad thing?
And the silence haunts our bed chamber
You make me do too much labour
You make me do too much labour
Apologies from my tongue, and never yours
Busy lapping from flowing cup and stabbing with your fork
I know you're a smart man (I know you're a smart man)
And weaponise
The false incompetence, it's dominance under a guise
If we had a daughter, I'd watch and could not save her
The emotional torture from the head of your high table
She'd do what you taught her
She'd meet the same cruel fate
So now I've gotta run, so I can undo this mistake
At least I've gotta try
The capillaries in my eyes are bursting
If our love died, would that be the worst thing?
For somebody I thought was my saviour
You sure make me do a whole lot of labour
The calloused skin on my hands is cracking
If our love ends, would that be a bad thing?
And the silence haunts our bed chamber
You make me do too much labour
All day, every day, therapist, mother, maid
Nymph, then a virgin, nurse, then a servant
Just an appendage, live to attend him
So that he never lifts a finger
24/7 baby machine
So he can live out his picket-fence dreams
It's not an act of love if you make her
You make me do too much labour
All day, every day, therapist, mother, maid
Nymph, then virgin, nurse, then a servant
Just an appendage, live to attend him
So that he never lifts a finger
24/7 baby machine
So he can live out his picket-fence dreams
It's not an act of love if you make her
You make me do too much labour
The capillaries in my eyes (all day, every day)
Are bursting (therapist, mother, maid)
If our love died (nymph, then virgin)
Would that be the worst thing? (Nurse, then a servant)
For somebody (just an appendage)
I thought was my saviour (live to attend him)
You sure make me do (so that)
A whole lot of labour (he never lifts a finger)
The calloused skin on my hands (24/7)
Is cracking (baby machine)
If our love ends (so he can live out)
Would that be a bad thing? (His picket-fence dreams)
And the silence (it's not an act of love)
Haunts our bed chamber (if you make her)
You make me do too much labour
Source: Musixmatch
"""

with open("/content/audio_test.txt", "w") as f:
    f.write(text_content)
print("✓ Text file created")

# Ingest it
count, mod = ingest_file( "/content/audio_test.txt")
print(f"✓ Ingested as {mod} — {count} chunks")
print(f"✓ Total docs in DB: {COLLECTION.count()}")

✓ Text file created
[TEXT] ingesting  /content/audio_test.txt
  ✓ stored 9 chunks
✓ Ingested as text — 9 chunks
✓ Total docs in DB: 22


In [16]:
print(f"Total docs in DB: {COLLECTION.count()}")
print()

# Show what's stored
sample = COLLECTION.peek(limit=10)
seen = {}
for meta in sample['metadatas']:
    mod = meta.get('modality')
    fname = meta.get('filename')
    key = f"{mod}:{fname}"
    if key not in seen:
        seen[key] = True
        icon = {"text":"📄","audio":"🎵","video":"🎬"}.get(mod,"📁")
        print(f"  {icon} [{mod}]  {fname}")

Total docs in DB: 22

  🎬 [video]  human_drinking_milk.mp4
  🎵 [audio]  xyz_band_audio.mp3


In [17]:
# For the 4th mode — you speak, Whisper transcribes, then it searches text docs
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav
import tempfile
import os # Ensure os is imported

# Removed: os.system("pip install -q sounddevice scipy") as it's already installed and redundant here.

def spoken_query(duration=5):
    RATE = 16000
    print(f"🎙️  Speak now — recording for {duration} seconds... (NOTE: Live recording often requires local runtime or browser API.)")

    # --- WORKAROUND for Colab environment: use a pre-recorded audio file ---
    # In a real application, this would be sd.rec(..., dtype='int16')
    # For demonstration, we'll use an existing audio file.
    audio_file_path = "/content/xyz_band_audio.mp3" # Using the previously generated audio
    if not os.path.exists(audio_file_path):
        print(f"✗ Audio file not found at {audio_file_path}. Please ensure it's created or specify a different path.")
        return

    print(f"  Using pre-recorded audio: {audio_file_path}")
    # Convert to WAV for Whisper if needed, or use directly if Whisper supports mp3
    # For simplicity, if Whisper supports mp3 directly, we can use it.
    # Assuming Whisper can handle mp3 directly, as per previous notebook cells.
    tmp_audio_path = audio_file_path # Direct use for Whisper for demonstration

    # sd.wait() # No wait needed for pre-recorded file
    print("✓ Audio selected. Transcribing...")

    # Transcribe with Whisper
    segments_iter, _ = WHISPER.transcribe(tmp_audio_path, beam_size=5)
    segments = list(segments_iter)
    spoken = " ".join(s.text for s in segments).strip()
    # os.unlink(tmp.name) # Not needed as we are using an existing file

    if not spoken:
        print("✗ Could not detect speech from file. Try again with a different file.")
        return

    print(f"✓ Transcribed text: \"{spoken}\"")
    print(f"  Searching text documents...")

    # Search only text docs
    hits = search(spoken, n_results=3, modality_filter="text")
    print(f"\n{'='*50}")
    print(f"AUDIO → TEXT RESULTS")
    print(f"Query: \"{spoken}\"")
    print(f"{'='*50}")
    for i, h in enumerate(hits, 1):
        print(f"\nResult {i}: 📄 score={h['score']}")
        print(f"  File:    {h['source']}")
        print(f"  Snippet: {h['content'][:150]}")

# Run it — speak something like "machine learning neural networks"
spoken_query(duration=5)


🎙️  Speak now — recording for 5 seconds... (NOTE: Live recording often requires local runtime or browser API.)
  Using pre-recorded audio: /content/xyz_band_audio.mp3
✓ Audio selected. Transcribing...
✓ Transcribed text: "XYZ is a famous music band from the 1990s. Their most popular song is called Midnight Dreams.  The lead singer of XYZ performed beautifully at the concert. XYZ released their debut album in  1995 and became worldwide famous. The song Midnight Dreams by XYZ reached number one on the charts."
  Searching text documents...

AUDIO → TEXT RESULTS
Query: "XYZ is a famous music band from the 1990s. Their most popular song is called Midnight Dreams.  The lead singer of XYZ performed beautifully at the concert. XYZ released their debut album in  1995 and became worldwide famous. The song Midnight Dreams by XYZ reached number one on the charts."

Result 1: 📄 score=0.1091
  File:    audio_test.txt
  Snippet: You sure make me do (so that)
A whole lot of labour (he never lifts a f

In [18]:
print(f"Total docs in DB: {COLLECTION.count()}")
sample = COLLECTION.peek(limit=20)
seen = {}
for meta in sample['metadatas']:
    key = f"{meta.get('modality')}:{meta.get('filename')}"
    if key not in seen:
        seen[key] = True
        icon = {"text":"📄","audio":"🎵","video":"🎬"}.get(meta.get('modality'),"📁")
        print(f"  {icon} {meta.get('filename')}")

Total docs in DB: 22
  🎬 human_drinking_milk.mp4
  🎵 xyz_band_audio.mp3
  📄 knowledge_base.txt
  📄 audio_test.txt


In [19]:
import os
from gtts import gTTS

# Video
if not os.path.exists("/content/human_drinking_milk.mp4"):
    tts = gTTS("A person is drinking a glass of cold milk. The human drinks milk every morning.", lang='en')
    tts.save("/tmp/milk.mp3")
    os.system("ffmpeg -i /tmp/milk.mp3 /tmp/milk.wav -y -loglevel quiet")
    os.system("ffmpeg -y -f lavfi -i 'color=c=white:size=640x360:rate=24' "
              "-i /tmp/milk.wav -shortest -c:v libx264 -c:a aac "
              "/content/human_drinking_milk.mp4 -loglevel quiet")
count, _ = ingest_file("/content/human_drinking_milk.mp4")
print(f"✓ Video: {count} chunks")

# Audio
if not os.path.exists("/content/xyz_band_audio.mp3"):
    tts2 = gTTS("XYZ is a famous band. Their song Midnight Dreams became a worldwide hit.", lang='en')
    tts2.save("/content/xyz_band_audio.mp3")
count, _ = ingest_file("/content/xyz_band_audio.mp3")
print(f"✓ Audio: {count} chunks")

# Text
with open("/content/knowledge_base.txt","w") as f:
    f.write("Machine learning models learn patterns from large datasets.\n"
            "Deep learning is a subset of machine learning using neural networks.\n"
            "Python is the most popular programming language for AI development.\n"
            "The recipe requires two eggs and one cup of flour.\n"
            "Artificial intelligence is transforming modern technology.\n")
count, _ = ingest_file("/content/knowledge_base.txt")
print(f"✓ Text: {count} chunks")

print(f"\n✓ Total docs in DB: {COLLECTION.count()}")
print("Go to the UI — top right should now show 'API connected' and doc count")

[VIDEO] ingesting  /content/human_drinking_milk.mp4
  Extracting frames …
  ✓ stored 5 frame captions
  Transcribing audio track …
  ✓ stored 3 audio segments
  Total stored for this video: 8
✓ Video: 8 chunks
[AUDIO] ingesting  /content/xyz_band_audio.mp3
  ✓ stored 3 audio segments
✓ Audio: 3 chunks
[TEXT] ingesting  /content/knowledge_base.txt
  ✓ stored 1 chunks
✓ Text: 1 chunks

✓ Total docs in DB: 22
Go to the UI — top right should now show 'API connected' and doc count


In [20]:
# =============================================================================
# CELL 5 (MOST RELIABLE FIX) — Colab built-in port forwarding
# No ngrok, no cloudflare, no external tools needed
# =============================================================================

import os, shutil, threading, time
import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse

nest_asyncio.apply()

# ── Kill anything on port 8000 first ─────────────────────────────────────────
os.system("fuser -k 8000/tcp 2>/dev/null")
time.sleep(2)
print("✓ Port 8000 cleared")

# ── Build FastAPI app ─────────────────────────────────────────────────────────
app = FastAPI(title="Multi-Modal Embedding API", version="1.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)

UPLOAD_DIR = "/content/uploads"
UI_FILE    = "/content/ui/index.html"
os.makedirs(UPLOAD_DIR, exist_ok=True)


@app.get("/", response_class=HTMLResponse)
def serve_ui():
    if not os.path.exists(UI_FILE):
        return HTMLResponse(
            "<h2 style='font-family:sans-serif;padding:2rem'>"
            "UI file not found.<br>"
            "<small>Upload index.html to /content/ui/</small></h2>",
            status_code=404
        )
    with open(UI_FILE, "r") as f:
        return HTMLResponse(f.read())


@app.get("/health")
def health():
    return {"status": "ok", "docs_stored": COLLECTION.count()}

# =============================================================================
# ADD THESE TWO ROUTES to your Cell 5 FastAPI app
# Paste them right after the /health route
# These make the Download buttons in the UI actually work
# =============================================================================

from fastapi.responses import FileResponse
import urllib.parse

# Folders where ingested files live
FILE_SEARCH_DIRS = [
    "/content/uploads",
    "/content",
    "/tmp",
]

@app.get("/files/{filename}")
def serve_file(filename: str):
    """
    Serve an ingested file for download/playback.
    The UI calls /files/<filename> when user clicks Download.
    """
    filename = urllib.parse.unquote(filename)

    # Strip any path prefix — only serve by base filename for security
    basename = os.path.basename(filename)

    # Search known directories
    for directory in FILE_SEARCH_DIRS:
        candidate = os.path.join(directory, basename)
        if os.path.exists(candidate):
            return FileResponse(
                path=candidate,
                filename=basename,
                media_type=_guess_media_type(basename)
            )

    raise HTTPException(status_code=404, detail=f"File not found: {basename}")


def _guess_media_type(filename: str) -> str:
    ext = os.path.splitext(filename)[1].lower()
    return {
        ".mp4":  "video/mp4",
        ".mov":  "video/quicktime",
        ".avi":  "video/x-msvideo",
        ".mkv":  "video/x-matroska",
        ".mp3":  "audio/mpeg",
        ".wav":  "audio/wav",
        ".m4a":  "audio/mp4",
        ".ogg":  "audio/ogg",
        ".txt":  "text/plain",
        ".md":   "text/markdown",
        ".pdf":  "application/pdf",
    }.get(ext, "application/octet-stream")


# Also add /list endpoint so UI can show all available files
@app.get("/list")
def list_files():
    """Return all files that have been ingested (for debugging)."""
    files = []
    for d in FILE_SEARCH_DIRS:
        if os.path.exists(d):
            for f in os.listdir(d):
                full = os.path.join(d, f)
                if os.path.isfile(full):
                    files.append({
                        "filename": f,
                        "path":     full,
                        "size_kb":  round(os.path.getsize(full) / 1024, 1),
                        "url":      f"/files/{f}"
                    })
    return {"files": files}
@app.post("/ingest")
async def api_ingest(file: UploadFile = File(...)):
    save_path = os.path.join(UPLOAD_DIR, file.filename)
    with open(save_path, "wb") as f:
        shutil.copyfileobj(file.file, f)
    try:
        count, modality = ingest_file(save_path)
        return {
            "status":        "ingested",
            "file":          file.filename,
            "modality":      modality,
            "chunks_stored": count
        }
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/search")
def api_search(q: str, n: int = 5, modality: str = None):
    if not q.strip():
        raise HTTPException(status_code=400, detail="Query cannot be empty.")
    hits = search(q, n_results=n, modality_filter=modality or None)
    return {"query": q, "total": len(hits), "results": hits}
# =============================================================================
# ADD THIS ROUTE to your Cell 5 FastAPI app — paste it after the /search route
# This powers the "Upload voice file" button in the UI
# =============================================================================

@app.post("/transcribe")
async def api_transcribe(file: UploadFile = File(...)):
    """
    Receive an audio file, transcribe it with Whisper,
    return the spoken text so the UI can search with it.
    """
    import tempfile, os

    # Save uploaded audio to temp file
    suffix = os.path.splitext(file.filename)[1] or ".wav"
    tmp = tempfile.NamedTemporaryFile(suffix=suffix, delete=False)
    shutil.copyfileobj(file.file, tmp)
    tmp.close()

    try:
        # Transcribe with faster-whisper
        segments_iter, info = WHISPER.transcribe(tmp.name, beam_size=5)
        segments = list(segments_iter)
        transcript = " ".join(s.text for s in segments).strip()

        if not transcript:
            raise HTTPException(
                status_code=422,
                detail="No speech detected in the audio file."
            )

        return {
            "transcript": transcript,
            "language":   info.language,
            "duration":   round(info.duration, 2)
        }

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        os.unlink(tmp.name)

@app.delete("/reset")
def api_reset():
    global COLLECTION
    chroma_client.delete_collection("multimodal_store")
    COLLECTION = chroma_client.get_or_create_collection(
        name="multimodal_store",
        metadata={"hnsw:space": "cosine"}
    )
    return {"status": "reset", "docs_stored": 0}


# ── Start FastAPI in background ───────────────────────────────────────────────
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=run_api, daemon=True)
thread.start()
time.sleep(3)
print("✓ FastAPI started")

# ── Verify server is actually running ─────────────────────────────────────────
import urllib.request
try:
    urllib.request.urlopen("http://localhost:8000/health", timeout=5)
    print("✓ Server responding on port 8000")
except Exception as e:
    print(f"✗ Server not responding: {e}")

# ── Get public URL using Colab's built-in proxy ───────────────────────────────
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8000)")

print("\n" + "=" * 60)
print("  ✓ YOUR APP URL (click or copy into browser):")
print(f"  {url}")
print("=" * 60)
print("\n  This URL only works while this Colab session is active.")
print("  If it stops working, re-run this cell to get a fresh URL.\n")

✓ Port 8000 cleared
✓ FastAPI started
✓ Server responding on port 8000

  ✓ YOUR APP URL (click or copy into browser):
  https://8000-m-s-kkb-ase1a1-2ly5ig62oyvl6-a.asia-east1-1.prod.colab.dev

  This URL only works while this Colab session is active.
  If it stops working, re-run this cell to get a fresh URL.

